# PyTorch Fine-tuning: Schritt für Schritt

**Estimated time: 45 minutes**

## Lernziele
- Den Training Loop verstehen
- Loss-Kurven interpretieren
- Hyperparameter tunen
- Checkpoints speichern und laden

## Übersicht
In diesem Notebook lernen Sie, wie man ein Large Language Model (LLM) mit PyTorch fine-tuned. Wir verwenden LoRA (Low-Rank Adaptation) für effizientes Training.

## Teil 1: Dataset Loading

Zuerst laden wir unser Dataset. Wir verwenden das Alpaca-Format für Instruction Following.

In [ ]:
import json
from pathlib import Path

# Dataset laden
dataset_path = "../datasets/student_chat.json"

with open(dataset_path) as f:
    data = json.load(f)

print(f"📚 Dataset geladen: {len(data)} Samples")
print(f"\nBeispiel:")
print(json.dumps(data[0], indent=2, ensure_ascii=False))

### 💡 Exercise 1: Dataset Exploration

**Aufgabe:** Analysieren Sie das Dataset
1. Wie viele Samples hat das Dataset?
2. Welche Felder hat jedes Sample?
3. Was ist die durchschnittliche Länge der Outputs?

In [ ]:
# TODO: Ihre Lösung hier

num_samples = len(data)
print(f"Anzahl Samples: {num_samples}")

# Durchschnittliche Output-Länge berechnen
avg_length = sum(len(item["output"]) for item in data) / len(data)
print(f"Durchschnittliche Output-Länge: {avg_length:.1f} Zeichen")

## Teil 2: Model Loading

Jetzt laden wir das Base Model und bereiten es für LoRA Fine-tuning vor.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Für dieses Tutorial verwenden wir GPT-2 (schneller)
# In Produktion würden Sie Llama-3.1-8B verwenden
model_name = "gpt2"

print(f"📥 Loading model: {model_name}")

# Tokenizer laden
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Model laden
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"✅ Model geladen")
print(f"   Parameter: {sum(p.numel() for p in model.parameters()):,}")

### LoRA Configuration

LoRA fügt kleine trainierbare Matrizen zu den Attention-Layern hinzu. Der `r`-Parameter steuert die Größe dieser Matrizen.

In [ ]:
# LoRA Config
lora_config = LoraConfig(
    r=8,  # Rank - bestimmt Anzahl trainierbare Parameter
    lora_alpha=16,  # Skalierungsfaktor
    target_modules=["c_attn"],  # Welche Module trainiert werden
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# LoRA zum Model hinzufügen
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### 💡 Exercise 2: LoRA Parameters

**Aufgabe:** Experimentieren Sie mit verschiedenen LoRA Ranks
- Was passiert bei `r=4`?
- Was passiert bei `r=16`?
- Wie ändert sich die Anzahl trainierbare Parameter?

In [ ]:
# TODO: Testen Sie verschiedene r-Werte

for r in [4, 8, 16]:
    test_config = LoraConfig(
        r=r,
        lora_alpha=16,
        target_modules=["c_attn"],
        task_type="CAUSAL_LM"
    )
    
    test_model = AutoModelForCausalLM.from_pretrained(model_name)
    test_model = get_peft_model(test_model, test_config)
    
    trainable = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in test_model.parameters())
    
    print(f"r={r}: {trainable:,} trainable params ({100*trainable/total:.2f}%)")

## Teil 3: Dataset Preparation

Wir erstellen einen PyTorch DataLoader für unser Training.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Format als Alpaca-Prompt
        if item.get("input", ""):
            prompt = f"### Instruction:\n{item['instruction']}\n\n### Input:\n{item['input']}\n\n### Response:\n{item['output']}"
        else:
            prompt = f"### Instruction:\n{item['instruction']}\n\n### Response:\n{item['output']}"
        
        # Tokenize
        encoded = self.tokenizer(
            prompt,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoded["input_ids"].squeeze(),
            "attention_mask": encoded["attention_mask"].squeeze(),
            "labels": encoded["input_ids"].squeeze()
        }

# Dataset und DataLoader erstellen
dataset = InstructionDataset(data, tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

print(f"📦 DataLoader erstellt: {len(dataloader)} batches")

## Teil 4: Training Loop

Jetzt implementieren wir den Trainingsloop.

In [ ]:
from torch.optim import AdamW
from tqdm.notebook import tqdm

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-4)

# Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()

num_epochs = 3
losses = []

print(f"🚀 Starting training for {num_epochs} epochs...\n")

for epoch in range(num_epochs):
    epoch_loss = 0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        
        # Backward pass
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        epoch_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}: Average Loss = {avg_loss:.4f}")

print("\n✅ Training completed!")

### Visualize Training Progress

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs + 1), losses, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Average Loss")
plt.title("Training Loss over Epochs")
plt.grid(True)
plt.show()

### 💡 Exercise 3: Loss Analysis

**Aufgabe:** Analysieren Sie die Loss-Kurve
1. Sinkt der Loss kontinuierlich?
2. Gibt es Anzeichen von Overfitting?
3. Sollten wir mehr Epochen trainieren?

## Teil 5: Checkpoint Saving

Speichern Sie das trainierte Model.

In [ ]:
# Checkpoint speichern
checkpoint_dir = "../checkpoints/tutorial_model"
Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)

model.save_pretrained(checkpoint_dir)
tokenizer.save_pretrained(checkpoint_dir)

print(f"💾 Checkpoint saved: {checkpoint_dir}")

## Teil 6: Inference Testing

Testen Sie das fine-getunte Model.

In [ ]:
# Model in eval mode
model.eval()

test_instruction = "Erkläre was ein neuronales Netz ist."
prompt = f"### Instruction:\n{test_instruction}\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated Response:")
print(response)

## Summary

### Was Sie gelernt haben:
1. ✅ Dataset laden und vorbereiten
2. ✅ Model mit LoRA konfigurieren
3. ✅ Training Loop implementieren
4. ✅ Loss-Kurven visualisieren
5. ✅ Checkpoints speichern
6. ✅ Fine-getunte Models testen

### Nächste Schritte:
- **Notebook 02:** LoRA im Detail
- **Notebook 03:** Distributed Training für größere Models

### Weitere Ressourcen:
- [PEFT Documentation](https://huggingface.co/docs/peft)
- [LoRA Paper](https://arxiv.org/abs/2106.09685)
- [Transformers Documentation](https://huggingface.co/docs/transformers)